<!--
Copyright (c) Meta Platforms, Inc. and affiliates.
All rights reserved.

This source code is licensed under the CC-BY-NC-4.0 license found in the
LICENSE file in the root directory of this source tree.
-->

In [ ]:
import os, sys
from pathlib import Path
REPO = Path(os.environ.get('LONG_AUDIO_REPO', Path.cwd())).resolve()
if REPO.name == 'notebooks':
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

# %% [markdown]
# # Ego4D Segmentation Output Visualizer
#
# Per-vid comparison of segmentation outputs across E2E + Cascade models.
# Mirror of compare_models_sins.py, adapted for Ego4D's per-uid layout:
#   E2E:     <RUN_DIR_E2E>/<arm>/ego4d/<uid>/chunk_*.json
#   Cascade: <RUN_DIR_CASC>/cascade_<arm>/<uid>/chunk_*.json
#
# Ego4D ships TWO annotator passes per vid (passes["1"], passes["2"]);
# both are rendered as separate GT rows so disagreement is visible.
#
# Dependencies: stdlib, numpy, matplotlib only.

# %% Imports
import json
import os
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Pick a single video to visualize. Either set UID directly (full UUID,
# not the trailing "ego4d_" prefix) or set UID=None and use VID_INDEX
# to pick by position in the test-split iteration order.
UID = None
VID_INDEX = 0


def _arms(parent: Path, prefix: str = ""):
    if not parent.is_dir():
        return []
    out = []
    for d in sorted(os.listdir(parent)):
        full = parent / d
        if not full.is_dir():
            continue
        if prefix and not d.startswith(prefix):
            continue
        out.append(d[len(prefix):] if prefix else d)
    return out

DATASET = "egolife"
ARMS = {
    "Casc. GPT 5.6": str(REPO / "runs/desc-v4-20260715T160821Z/cascB_luna/cascA_af3-hf"),
    "Casc. Gemma 4": str(REPO / "runs/desc-v4-20260715T160821Z/cascB_gemma4-31b/cascA_af3-hf"),
    "Casc. Gemma 4 + SFT": str(REPO / "runs/desc-v4-20260715T160821Z/cascB_gemma4-31b_ft/cascA_af3-hf"),
    "E2E Gemini 3.5": str(REPO / "runs/desc-v4-20260715T160821Z/e2e_gemini"),
    "E2E Qwen 3-o": str(REPO / "runs/desc-v4-20260715T160821Z/e2e_qwen3-omni"),
    "E2E Qwen 3-o + SFT": str(REPO / "runs/desc-v4-20260715T160821Z/e2e_qwen3-omni_ft"),
}

# %% Load Ego4D test split via Ego4DDataset (for iteration order) +
# raw manifest (for per-pass GT — iter_items collapses to a single list)
import sys

if DATASET == "ego4d":
    from long_audio.datasets.ego4d.dataset import Ego4DDataset
    MANIFEST_PATH = REPO / "datasets/ego4d/annotated_manifest.json"
    _ds = Ego4DDataset.eval_set(str(MANIFEST_PATH), split="test")
elif DATASET == "egolife":
    from long_audio.datasets.egolife.dataset import EgoLifeDataset
    MANIFEST_PATH = REPO / "datasets/egolife/manifest.json"
    _ds = EgoLifeDataset.eval_set(str(MANIFEST_PATH))
else:
    raise Exception

_items = list(_ds.iter_items())
_manifest_videos = {v["uid"]: v for v in json.loads(MANIFEST_PATH.read_text())["videos"]}
print(f"test split size: {len(_items)}")


def _uid(it):
    return getattr(it, "id", "").removeprefix("ego4d_").removeprefix("egolife_")

# %% [markdown]
# ## Video chooser
# Pick a vid from the dropdown (idx | uid-prefix | duration | GT segs/pass).
# Updates the UID global; re-run the "Resolve UID" cell + downstream
# cells to refresh plots for the selected vid.

# %%
try:
    import ipywidgets as widgets
    from IPython.display import display

    _options = []
    for i, it in enumerate(_items):
        u = _uid(it)
        v = _manifest_videos.get(u, {})
        per_pass = ", ".join(
            f"p{k}={len(pd.get('actions', []))}"
            for k, pd in v.get("passes", {}).items()
        )
        dur_min = getattr(it, "duration", 0.0) / 60.0
        label = f"[{i:3d}] {u[:8]}…  {dur_min:5.1f} min  GT={per_pass}"
        _options.append((label, u))

    _initial = UID if UID in [u for _, u in _options] else _options[VID_INDEX][1]
    _picker = widgets.Dropdown(
        options=_options, value=_initial, description="vid:",
        layout=widgets.Layout(width="600px"),
        style={"description_width": "40px"},
    )

    def _on_pick(change):
        global UID
        UID = change["new"]
        print(f"UID set to {UID} — re-run cells below to refresh plots.")

    _picker.observe(_on_pick, names="value")
    display(_picker)
    UID = _picker.value
except ImportError:
    print("ipywidgets not available — set UID or VID_INDEX manually and re-run.")


# %% [markdown]
# ## Timeline Strip Plot

# %%
def plot_strips(models, model_preds, gt_events=None,
                start_min=0.0, end_min=None, title="Segmentation Timeline"):
    if end_min is None:
        end_min = duration_s / 60.0
    start_s = start_min * 60.0
    end_s = end_min * 60.0

    rows = []
    row_labels = []
    if gt_events:
        for lbl, segs in gt_events.items():
            rows.append(segs)
            row_labels.append(lbl)
    for model in models:
        rows.append(model_preds[model])
        row_labels.append(model)

    fig, axes = plt.subplots(len(rows), 1, figsize=(14, 0.5 * len(rows)), sharex=True)
    if len(rows) == 1:
        axes = [axes]

    for ax, events, label in zip(axes, rows, row_labels):
        for e in events:
            s = float(e["start"])
            en = float(e["end"])
            if en <= start_s or s >= end_s:
                continue
            s = max(s, start_s)
            en = min(en, end_s)
            color = label_colors.get(e["label"], (0.5, 0.5, 0.5, 1.0))
            ax.axvspan(s / 60.0, en / 60.0, color=color, linewidth=0)
        ax.set_xlim(start_min, end_min)
        ax.set_yticks([])
        ax.set_ylabel(label, fontsize=9, rotation=0, ha="right", va="center")

    axes[-1].set_xlabel("Time (minutes)")
    if title:
        axes[0].set_title(f"{title}  —  {UID[:8]}…", fontsize=12)

    patches = [mpatches.Patch(color=label_colors[l], label=l) for l in all_labels]
    fig.legend(handles=patches, loc="lower center",
               ncol=min(8, len(patches)), bbox_to_anchor=(0.5, -0.05), fontsize=8)
    plt.tight_layout()
    plt.show()

In [ ]:
# %% Resolve UID + load both passes' GT
if UID is None:
    if VID_INDEX >= len(_items):
        raise IndexError(f"VID_INDEX={VID_INDEX} out of range (test split has {len(_items)})")
    UID = _uid(_items[VID_INDEX])

_vid = _manifest_videos.get(UID)
if _vid is None:
    raise ValueError(f"UID {UID} not found in manifest")

duration_s = float(_vid["duration"])
print(f"UID: {UID}")
print(f"Duration: {duration_s:.1f}s ({duration_s/60:.1f} min)")

# gt_events: ordered dict {label -> [segments]}, one entry per pass.
gt_events: dict[str, list[dict]] = {}
for pass_num, pass_data in sorted(_vid.get("passes", {}).items()):
    gt_events[f"GT-pass{pass_num}"] = [
        {"label": a["event"], "start": float(a["start"]), "end": float(a["end"])}
        for a in pass_data.get("actions", [])
    ]
for lbl, segs in gt_events.items():
    print(f"  {lbl}: {len(segs)} segments")

# %% Load predictions per arm
def load_predictions(base: Path, uid: str):
    chunks = sorted((base / DATASET / uid).glob("chunk_*.json"))
    all_segs = []
    for cf in chunks:
        rec = json.loads(cf.read_text())
        all_segs.extend(rec.get("segments_abs", []))
    return all_segs


model_preds: dict[str, list[dict]] = {
    key: load_predictions(Path(value), UID)
    for key, value in ARMS.items()
}
for name, segs in model_preds.items():
    print(f"  {name}: {len(segs)} segments")

# %% Build color map (across GT passes + all preds)
all_labels = set()
for preds in model_preds.values():
    all_labels.update(e["label"] for e in preds)
for segs in gt_events.values():
    all_labels.update(e["label"] for e in segs)
all_labels = sorted(all_labels)

cmap = plt.cm.tab20(np.linspace(0, 1, max(1, len(all_labels))))
label_colors = {label: cmap[i] for i, label in enumerate(all_labels)}
print(f"Labels ({len(all_labels)}): {all_labels}")

In [ ]:
plot_strips(list(model_preds), model_preds, gt_events, title="Full video")

In [ ]:
!ls {REPO}/runs/desc-v4-20260715T160821Z/e2e_qwen3-omni_ft/egolife/A5_KATRINA_DAY1_S03